<a href="https://colab.research.google.com/github/hfelizzola/Diplomado_Analitica_DPI/blob/main/Modulo1_Analisis_Credito.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Módulo 1 · Fundamentos de Analítica de Datos
## Exploración, diagnóstico, limpieza y análisis exploratorio

**Diplomado en Análisis de Datos con Herramienta Tableau**
Universidad de La Salle · Heriberto Felizzola PhD.

---

## Bloque 0 · Contexto y diccionario de datos

### El escenario

Una entidad financiera colombiana quiere entender el perfil de sus
solicitantes de crédito de consumo y evaluar si existen patrones en
las aprobaciones y los rechazos. Para ello, exportó de su sistema un
archivo con todas las solicitudes radicadas entre **enero de 2024 y
junio de 2025**.

El archivo tiene alrededor de **3.000 registros y 13 columnas**. Cada
fila representa una solicitud de crédito. Cada columna describe una
característica de la solicitud o del solicitante.

El archivo llegó tal cual lo exportaron: **no ha sido limpiado ni
revisado**. Antes de construir cualquier tablero o reporte, necesitamos
entender qué contiene, evaluar su calidad y prepararlo para el análisis.

---

### Diccionario de datos

| Columna | Tipo esperado | Descripción | Valores válidos |
|---|---|---|---|
| `id_solicitud` | Identificador | Código único de la solicitud | SOL-00001 a SOL-03000 |
| `fecha_solicitud` | Temporal | Fecha en que se radicó la solicitud | Enero 2024 – Junio 2025 |
| `sucursal` | Nominal / Geográfico | Ciudad de la sucursal que recibió la solicitud | 7 ciudades colombianas |
| `tipo_credito` | Nominal | Categoría del crédito solicitado | Libre inversión, Vehículo, Vivienda, Educación |
| `monto_solicitado` | Continuo | Valor del crédito en pesos colombianos | Depende del tipo de crédito |
| `plazo_meses` | Discreto | Duración pactada del crédito en meses | 12, 24, 36, 48, 60, 120, 180, 240 |
| `ingresos_mensuales` | Continuo | Ingreso mensual declarado por el solicitante | > 0 |
| `antiguedad_laboral` | Discreto | Años en el empleo actual | 0 en adelante |
| `nivel_educativo` | Ordinal | Último nivel de estudios alcanzado | Bachiller, Técnico, Profesional, Posgrado |
| `estado_civil` | Nominal | Estado civil del solicitante | Soltero, Casado, Unión libre, Divorciado |
| `numero_dependientes` | Discreto | Personas que dependen económicamente del solicitante | 0 en adelante |
| `tiene_otros_creditos` | Nominal | Si el solicitante tiene créditos vigentes con otra entidad | Sí, No |
| `resultado` | Nominal | Decisión de la entidad sobre la solicitud | Aprobado, Rechazado, En estudio |

---

### ¿Por qué empezamos por aquí?

Antes de escribir una sola línea de código, necesitamos saber:

1. **Qué representa cada fila** (una solicitud, no un cliente: un cliente puede tener varias).
2. **Qué debería contener cada columna** (el diccionario de datos).
3. **Qué rango de valores tiene sentido** (un ingreso de cero no tiene sentido; un plazo de 240 meses sí, si es vivienda).

Sin este contexto, cualquier hallazgo posterior se queda sin interpretación.

### Cómo subir el archivo a Google Colab

Use el ícono de carpeta en la barra lateral izquierda de Colab, o ejecute
la celda siguiente para subirlo desde su computador.

In [14]:
# Solo necesario en Google Colab
from google.colab import files
archivos = files.upload()

---
---
# Bloque 1 · Exploración y reconocimiento

Antes de tocar un solo dato, hay que entender qué tenemos entre manos.
Cada función que usemos aquí responde una pregunta concreta sobre el archivo.

## 1.1 · Cargar las librerías y el archivo

`pandas` es la librería principal para trabajar con tablas en Python.
`numpy` complementa con operaciones numéricas.
`matplotlib` permite crear gráficos básicos.

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Opciones de visualización para que las tablas se vean completas
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

`pd.read_csv()` lee un archivo CSV y lo convierte en un DataFrame: una tabla
con filas y columnas que podemos consultar, filtrar y transformar.

In [16]:
datos = pd.read_csv("solicitudes_credito.csv")

## 1.2 · ¿Qué tamaño tiene el archivo?

`.shape` devuelve una pareja: (número de filas, número de columnas).
Es lo primero que revisamos: nos dice con cuántos registros y cuántas
variables estamos trabajando.

In [17]:
datos.shape

(3045, 13)

## 1.3 · ¿Cómo se llaman las columnas?

`.columns` muestra la lista de nombres de las columnas, en el orden en que
aparecen en el archivo. Aquí verificamos que los nombres coincidan con el
diccionario de datos.

In [18]:
datos.columns.tolist()

['id_solicitud',
 'fecha_solicitud',
 'sucursal',
 'tipo_credito',
 'monto_solicitado',
 'plazo_meses',
 'ingresos_mensuales',
 'antiguedad_laboral',
 'nivel_educativo',
 'estado_civil',
 'numero_dependientes',
 'tiene_otros_creditos',
 'resultado']

## 1.4 · ¿Cómo lucen las primeras filas?

`.head(n)` muestra las primeras *n* filas. Es la forma más rápida de hacerse
una idea del contenido real del archivo: qué tipo de valores aparecen,
si hay mezcla de formatos y si algo salta a la vista.

In [19]:
datos.head(10)

,id_solicitud,fecha_solicitud,sucursal,tipo_credito,monto_solicitado,plazo_meses,ingresos_mensuales,antiguedad_laboral,nivel_educativo,estado_civil,numero_dependientes,tiene_otros_creditos,resultado
0,SOL-02907,2025-03-22,Bogota,Vehiculo,45700000,36,3350000,5,BACHILLER,casado,2.0,si,aprobado
1,SOL-02220,2024-04-21,Bucaramanga,Educacion,8000000,36,3270000,3,Tecnico,Union libre,2.0,No,Rechazado
2,SOL-02125,2024-04-13,barranquilla,Libre inversion,19600000,36,4270000,11,Profesional,Soltero,2.0,1,Rechazado
3,SOL-00604,02/07/2024,CARTAGENA,Libre inversion,20400000,36,4560000,6,Bachiller,Union libre,3.0,Si,rechazado
4,SOL-00563,2025-05-07,cartagena,Libre inversion,23800000,36,5.820.000,1,Posgrado,Divorciado,NaN,0,Aprobado
5,SOL-02269,04/12/2024,cali,Educacion,6800000,24,2650000,1,Tecnico,Union libre,NaN,No,Aprobado
6,SOL-01403,2025-04-29,Bucaramanga,Educacion,12600000,24,2040000,1,tecnico,Soltero,0.0,No,Aprobado
7,SOL-02068,2024-12-29,MEDELLIN,Vehiculo,36800000,24,4190000,6,Bachiller,Union libre,0.0,No,Aprobado
8,SOL-00733,15/11/2024,Bogotá D.C.,Vivienda,179200000,60,2350000,3,TECNICO,Soltero,2.0,No,Aprobado
9,SOL-00728,2025-03-04,medellin,Educacion,17300000,36,6200000,1,posgrado,divorciado,3.0,Si,aprobado


## 1.5 · ¿Qué tipo de dato leyó Python en cada columna?

`.info()` muestra tres cosas para cada columna:
- El nombre
- Cuántos valores no nulos tiene
- El tipo de dato que pandas asignó automáticamente (`int64`, `float64`, `object`)

In [20]:
datos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3045 entries, 0 to 3044
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id_solicitud          3045 non-null   object 
 1   fecha_solicitud       3045 non-null   object 
 2   sucursal              3045 non-null   object 
 3   tipo_credito          3045 non-null   object 
 4   monto_solicitado      3045 non-null   int64  
 5   plazo_meses           3045 non-null   int64  
 6   ingresos_mensuales    3020 non-null   object 
 7   antiguedad_laboral    2954 non-null   object 
 8   nivel_educativo       3027 non-null   object 
 9   estado_civil          3045 non-null   object 
 10  numero_dependientes   2937 non-null   float64
 11  tiene_otros_creditos  3045 non-null   object 
 12  resultado             3045 non-null   object 
dtypes: float64(1), int64(2), object(10)
memory usage: 309.4+ KB


### ¿Cómo leer los tipos de datos que asigna pandas?

Cuando ejecutamos `.info()`, pandas muestra en la última columna el **tipo de dato** que asignó automáticamente a cada variable. Estos son los más comunes:

| Tipo en pandas | Qué significa | Ejemplo en nuestro archivo |
|---|---|---|
| `int64` | Número entero (sin decimales) | `monto_solicitado`, `plazo_meses` |
| `float64` | Número con decimales | Aparece cuando una columna numérica tiene valores faltantes (`NaN`) |
| `object` | Texto (cadena de caracteres) | `sucursal`, `tipo_credito`, `resultado` |
| `datetime64` | Fecha y/o hora | Aparecerá después de convertir `fecha_solicitud` |
| `bool` | Verdadero o Falso | Aparecerá si creamos variables indicadoras (Sí/No) |
| `category` | Categoría con valores fijos | Se puede asignar manualmente para variables ordinales |

#### Lo que hay que vigilar

- **Una columna numérica que aparece como `object`** significa que al menos un valor dentro de ella no es un número. Puede ser un símbolo (`$`), un separador de miles (`.`), un código de vacío (`N/A`) o un espacio. Eso es exactamente lo que pasa con `ingresos_mensuales` y `antiguedad_laboral` en nuestro archivo.

- **Una columna de fecha que aparece como `object`** significa que pandas no la reconoció como fecha. Ocurre cuando hay formatos mezclados (como `2024-05-15` y `15/05/2024` en la misma columna).

- **`float64` en vez de `int64`** aparece cuando una columna de enteros tiene valores faltantes. Python no puede guardar un `NaN` en una columna de enteros, así que la convierte a decimal. Es lo que pasa con `numero_dependientes`.

> **Regla práctica:** si el tipo que muestra `.info()` no coincide con el tipo que dice el diccionario de datos, hay un problema de calidad que resolver antes de analizar.

**Deténgase aquí y compare con el diccionario de datos.**
¿Qué columnas tienen un tipo distinto al esperado? ¿Por qué cree
que ocurrió?

## 1.6 · ¿Cuántos valores distintos tiene cada columna?

`.nunique()` cuenta cuántos valores únicos hay en cada columna.
Un número mucho mayor al esperado es señal de inconsistencia:
si `sucursal` debería tener 7 ciudades y muestra 24, algo está escrito
de varias formas.

In [21]:
datos.nunique()

,0
id_solicitud,3000
fecha_solicitud,1021
sucursal,24
tipo_credito,4
monto_solicitado,1007
plazo_meses,8
ingresos_mensuales,1078
antiguedad_laboral,33
nivel_educativo,17
estado_civil,13


## 1.7 · ¿Qué contiene cada columna categórica?

`.value_counts()` cuenta cuántas veces aparece cada valor distinto.
Es la herramienta más útil para detectar inconsistencias de texto.

In [22]:
# Registros por sucursal
datos["sucursal"].value_counts()

,count
sucursal,
Bogota,514
Medellin,354
Cali,268
Barranquilla,255
Cartagena,156
Bucaramanga,155
Pereira,123
Bogotá D.C.,111
CALI,102


In [23]:
datos["nivel_educativo"].value_counts()

,count
nivel_educativo,
Profesional,651
Tecnico,532
Bachiller,507
Posgrado,284
Prof.,127
PROFESIONAL,123
profesional,102
Bach.,94
bachiller,92


In [24]:
datos["tiene_otros_creditos"].value_counts()

,count
tiene_otros_creditos,
No,1242
Si,854
no,190
NO,179
0,175
1,109
Sí,104
SI,102
si,90


## 1.8 · ¿Cómo se ven las variables numéricas?

`.describe()` genera un resumen estadístico: conteo, media, desviación
estándar, mínimo, cuartiles y máximo. Solo aplica a columnas numéricas.

In [12]:
datos.describe().round(0)

,monto_solicitado,plazo_meses,numero_dependientes
count,3045.0,3045.0,2937.0
mean,50675402.0,56.0,1.0
std,61096983.0,57.0,1.0
min,3000000.0,12.0,0.0
25%,14000000.0,24.0,0.0
50%,22200000.0,36.0,1.0
75%,52400000.0,60.0,2.0
max,249900000.0,240.0,4.0


**Observe:** `ingresos_mensuales` y `antiguedad_laboral` no aparecen en el
resumen. ¿Por qué? Porque pandas las leyó como texto (`object`), no como
número. Lo corregiremos en el Bloque 3.

---
### Tarea 1.A · Para hacer en clase

Explore las columnas que no revisamos juntos. Responda en las celdas
siguientes:

**Pregunta 1:** ¿Cuántos valores distintos tiene `estado_civil`?
¿Alguno parece estar escrito de más de una forma?

In [ ]:
# Escriba su código aquí

**Pregunta 2:** Use `.describe()` sobre `plazo_meses`. ¿El mínimo y
el máximo tienen sentido para un crédito de consumo? ¿Qué tipo de
crédito explica los plazos más largos?

In [ ]:
# Escriba su código aquí

**Pregunta 3:** ¿Cuántos valores distintos tiene `resultado`? Muestre
los conteos de cada uno. ¿Hay variantes de escritura?

In [ ]:
# Escriba su código aquí

---
---
# Bloque 2 · Diagnóstico de calidad

Ahora que conocemos el archivo, vamos a medir su calidad de forma
sistemática. El diagnóstico responde una pregunta por cada dimensión:
completitud, consistencia, validez, exactitud y unicidad.

## 2.1 · Unificar los códigos de valor vacío

En este archivo, un dato faltante puede aparecer como celda vacía,
como `N/A` o como un espacio en blanco. Para Python, esas tres cosas
son texto válido: las cuenta como si fueran datos presentes.

El primer paso es convertirlas todas a un vacío real (`NaN`) para que
las funciones de conteo las detecten correctamente.

In [ ]:
codigos_vacios = ["", " ", "N/A"]

datos = datos.replace(codigos_vacios, np.nan)

## 2.2 · Medir la completitud real

`.isna()` marca con `True` cada celda que está vacía.
`.sum()` cuenta cuántas hay por columna.
`.mean()` devuelve la proporción, que multiplicamos por 100 para verla
como porcentaje.

In [ ]:
datos.isna().sum()

In [ ]:
# En porcentaje
(datos.isna().mean() * 100).round(1)

Ahora sí sabemos cuántos datos faltan de verdad. Antes de este paso,
Python reportaba muchos menos porque los códigos de texto se contaban
como datos presentes.

## 2.3 · Revisar la cardinalidad de las categóricas

Cardinalidad = número de valores distintos. Comparamos lo que
encontramos con lo que esperamos según el diccionario de datos.

In [13]:
cols_categoricas = ["sucursal", "tipo_credito", "nivel_educativo",
                    "estado_civil", "tiene_otros_creditos", "resultado"]

for col in cols_categoricas:
    n = datos[col].nunique()
    print(f"{col:25s}  {n:>3} valores distintos")

sucursal                    24 valores distintos
tipo_credito                 4 valores distintos
nivel_educativo             17 valores distintos
estado_civil                13 valores distintos
tiene_otros_creditos         9 valores distintos
resultado                    9 valores distintos


Si `sucursal` debería tener 7 y encontramos 24, hay 17 variantes de
escritura por resolver. Si `tiene_otros_creditos` debería tener 2 y
encontramos 9, la codificación no es uniforme.

## 2.4 · Detectar duplicados

`.duplicated()` marca como `True` cada fila que ya apareció antes de
forma idéntica (mismas 13 columnas con los mismos valores).

In [ ]:
datos.duplicated().sum()

In [ ]:
# Veamos algunos de ellos
datos[datos.duplicated(keep=False)].sort_values("id_solicitud").head(6)

## 2.5 · Revisar rangos de las numéricas

Para las columnas que sí se leyeron como número, verificamos que los
valores estén dentro de un rango razonable.

In [ ]:
datos["monto_solicitado"].describe().round(0)

In [ ]:
datos["plazo_meses"].value_counts().sort_index()

## 2.6 · Identificar columnas numéricas leídas como texto

Revisamos los tipos actuales y los comparamos con lo que dice el
diccionario de datos.

In [ ]:
datos.dtypes

Las columnas `ingresos_mensuales`, `antiguedad_laboral` y
`numero_dependientes` aparecen como `object` (texto). Deberían ser
numéricas. Las convertiremos en el Bloque 3.

## 2.7 · Resumen del diagnóstico

Antes de limpiar, documentamos lo que encontramos:

| Dimensión | Hallazgo |
|---|---|
| **Completitud** | Cuatro columnas con datos faltantes; `antiguedad_laboral` es la más afectada |
| **Consistencia** | `sucursal`, `nivel_educativo`, `tiene_otros_creditos`, `estado_civil` y `resultado` tienen variantes de escritura |
| **Validez** | Tres columnas numéricas llegaron como texto por símbolos y separadores |
| **Unicidad** | 45 filas exactamente duplicadas |
| **Exactitud** | Los rangos de `monto_solicitado` y `plazo_meses` son coherentes con los tipos de crédito |

---
### Tarea 2.A · Para hacer en clase

**Pregunta 1:** Calcule el porcentaje de faltantes de cada columna
y ordénelos de mayor a menor. ¿El campo con más faltantes es un
campo obligatorio o uno que la gente podría no saber?

In [ ]:
# Escriba su código aquí

**Pregunta 2:** Muestre los valores únicos de `nivel_educativo`.
¿Cuántas formas distintas encontró para decir lo mismo? Escriba
en un comentario las equivalencias que usaría.

In [ ]:
# Escriba su código aquí

**Pregunta 3:** ¿Hay duplicados parciales? Es decir, ¿filas que
tengan el mismo `id_solicitud` pero valores distintos en otras
columnas? Use `.duplicated(subset=...)` para investigarlo.

In [ ]:
# Escriba su código aquí

---
---
# Bloque 3 · Limpieza

Corregimos los problemas en un orden fijo. El orden importa: si intenta
convertir tipos antes de limpiar el texto, la conversión falla.

El orden que seguiremos es:
1. Estandarizar texto (espacios, mayúsculas)
2. Homologar categorías con diccionarios
3. Convertir tipos (fechas, números)
4. Tratar valores faltantes
5. Eliminar duplicados

## 3.1 · Estandarizar el texto

`.str.strip()` quita espacios sobrantes al inicio y al final.
`.str.upper()` lleva todo a mayúsculas para que las comparaciones funcionen.

Esto lo aplicamos a todas las columnas de texto.

In [ ]:
columnas_texto = ["sucursal", "tipo_credito", "nivel_educativo",
                  "estado_civil", "tiene_otros_creditos", "resultado"]

for col in columnas_texto:
    datos[col] = datos[col].str.strip().str.upper()

In [ ]:
# Verificamos cuánto mejoró la sucursal
datos["sucursal"].value_counts()

Bajamos de muchas variantes a menos. Pero todavía quedan diferencias
que `.upper()` no resuelve: tildes y nombres alternativos.

## 3.2 · Quitar tildes

Las tildes hacen que «BOGOTÁ» y «BOGOTA» sean valores distintos para
la herramienta, aunque para nosotros son la misma ciudad.

In [ ]:
reemplazos = {"Á": "A", "É": "E", "Í": "I", "Ó": "O", "Ú": "U"}

for col in columnas_texto:
    for viejo, nuevo in reemplazos.items():
        datos[col] = datos[col].str.replace(viejo, nuevo, regex=False)

In [ ]:
datos["sucursal"].value_counts()

## 3.3 · Homologar categorías con diccionarios

Lo que queda son nombres alternativos legítimos y abreviaturas. Esto
no se resuelve con una función de texto: hay que decidir manualmente
qué corresponde a qué.

Un **diccionario de homologación** es una tabla que dice: «cuando
encuentres este valor, cámbialo por este otro».

In [ ]:
# Sucursal: la única variante que queda después de UPPER y sin tildes
homologar_sucursal = {
    "BOGOTA D.C.": "BOGOTA",
}

datos["sucursal"] = datos["sucursal"].replace(homologar_sucursal)
datos["sucursal"].value_counts()

Siete ciudades. Ahora sí el conteo es confiable.

In [ ]:
# Nivel educativo: abreviaturas y variantes
homologar_nivel = {
    "BACH.": "BACHILLER",
    "TEC.": "TECNICO",
    "PROF.": "PROFESIONAL",
    "POSTGRADO": "POSGRADO",
    "TÉCNICO": "TECNICO",
}

datos["nivel_educativo"] = datos["nivel_educativo"].replace(homologar_nivel)
datos["nivel_educativo"].value_counts()

In [ ]:
# Estado civil
homologar_civil = {
    "UNION LIBRE": "UNION LIBRE",  # ya está OK después de UPPER
}
# En este caso, UPPER ya resolvió casi todo. Verificamos:
datos["estado_civil"].value_counts()

In [ ]:
# Resultado
datos["resultado"].value_counts()

Después de pasar a mayúsculas y quitar tildes, `estado_civil` y
`resultado` quedaron limpios sin necesidad de diccionario adicional.

---
### Tarea 3.A · Para hacer en clase

**Construya el diccionario de homologación** para `tiene_otros_creditos`
y aplíquelo. Verifique con `.value_counts()` que solo queden dos
valores: SI y NO.

*Pista: revise los valores únicos actuales primero.*

In [ ]:
# Escriba su código aquí

## 3.4 · Convertir la fecha

La columna `fecha_solicitud` trae dos formatos mezclados: `2024-05-15`
y `15/05/2024`. El parámetro `format="mixed"` le dice a pandas que
intente interpretar cada valor según el formato que encuentre.
`dayfirst=True` indica que cuando haya ambigüedad, el primer número
es el día. `errors="coerce"` convierte a vacío lo que no pueda interpretar.

In [ ]:
datos["fecha_solicitud"] = pd.to_datetime(
    datos["fecha_solicitud"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

datos["fecha_solicitud"].head(10)

In [ ]:
# Verificamos el rango de fechas
datos["fecha_solicitud"].min(), datos["fecha_solicitud"].max()

## 3.5 · Convertir los ingresos a número

La columna `ingresos_mensuales` trae valores como `$4.560.000` o
`$ 3350000`. Antes de convertir, quitamos los símbolos de peso, los
puntos separadores de miles y los espacios.

In [ ]:
datos["ingresos_mensuales"] = (
    datos["ingresos_mensuales"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(" ", "", regex=False)
)

datos["ingresos_mensuales"] = pd.to_numeric(
    datos["ingresos_mensuales"], errors="coerce"
)

datos["ingresos_mensuales"].describe().round(0)

## 3.6 · Convertir antigüedad y dependientes

Estas dos columnas se leyeron como texto porque contenían valores
como `N/A` y espacios. Ahora que los reemplazamos por `NaN` en el
paso 2.1, la conversión funciona directamente.

In [ ]:
datos["antiguedad_laboral"] = pd.to_numeric(
    datos["antiguedad_laboral"], errors="coerce"
)

datos["numero_dependientes"] = pd.to_numeric(
    datos["numero_dependientes"], errors="coerce"
)

datos[["antiguedad_laboral", "numero_dependientes"]].describe().round(1)

---
### Tarea 3.B · Para hacer en clase

**Pregunta:** Después de las conversiones, ejecute `.info()` y compare
con el resultado del Bloque 1. ¿Qué columnas cambiaron de tipo?
¿Quedan columnas con el tipo incorrecto?

In [ ]:
# Escriba su código aquí

## 3.7 · Tratar valores faltantes

Revisamos cuántos faltan después de las conversiones y decidimos qué
hacer con cada uno.

In [ ]:
(datos.isna().mean() * 100).round(1)

Las decisiones:

- `antiguedad_laboral` (≈5%): imputamos con la **mediana**, que es más
  robusta que la media ante distribuciones sesgadas.
- `numero_dependientes` (≈4%): imputamos con la **moda** (el valor más
  frecuente), que es la opción natural para un dato discreto.
- `ingresos_mensuales` (<1%): imputamos con la **mediana**.
- `nivel_educativo` (<1%): imputamos con la **moda**.

In [ ]:
datos["antiguedad_laboral"] = datos["antiguedad_laboral"].fillna(
    datos["antiguedad_laboral"].median()
)

datos["numero_dependientes"] = datos["numero_dependientes"].fillna(
    datos["numero_dependientes"].mode()[0]
)

datos["ingresos_mensuales"] = datos["ingresos_mensuales"].fillna(
    datos["ingresos_mensuales"].median()
)

datos["nivel_educativo"] = datos["nivel_educativo"].fillna(
    datos["nivel_educativo"].mode()[0]
)

In [ ]:
# Verificamos que no queden faltantes
datos.isna().sum()

## 3.8 · Eliminar duplicados

In [ ]:
antes = len(datos)
datos = datos.drop_duplicates()
despues = len(datos)

print(f"Filas antes: {antes}")
print(f"Filas después: {despues}")
print(f"Eliminadas: {antes - despues}")

## 3.9 · Guardar el archivo limpio

In [ ]:
datos.to_csv("solicitudes_credito_limpio.csv", index=False, encoding="utf-8")
datos.shape

---
### Tarea 3.C · Para hacer en clase

**Pregunta:** Haga un resumen de las decisiones de limpieza. Para cada
problema que corregimos, escriba en un comentario:

1. Qué encontramos
2. Qué decidimos hacer
3. Cuántos registros se vieron afectados
4. Qué riesgo tiene esa decisión

In [ ]:
# Escriba su resumen aquí como comentarios
# Ejemplo:
# Problema: sucursal con 24 variantes de escritura
# Decisión: estandarizar a mayúsculas, quitar tildes y homologar "BOGOTA D.C." a "BOGOTA"
# Registros afectados: ~40% del archivo
# Riesgo: ninguno, todas las variantes referían a la misma ciudad

---
---
# Bloque 4 · Análisis exploratorio

Con el archivo limpio, ahora sí podemos analizarlo. El objetivo es
responder preguntas de negocio usando las herramientas básicas de la
estadística descriptiva.

## 4.1 · Resumen numérico general

`.describe()` aplicado al archivo limpio nos da el panorama completo
de todas las variables numéricas.

In [ ]:
datos.describe().round(0)

## 4.2 · Media, mediana y moda

Las tres describen «el centro» de una variable, pero responden
preguntas distintas:

- **Media:** promedio aritmético. Sensible a valores extremos.
- **Mediana:** el valor que parte el conjunto en dos mitades. Resistente.
- **Moda:** el valor más frecuente. La única que sirve para categóricas.

In [ ]:
print("Media de ingresos:  ", datos["ingresos_mensuales"].mean().round(0))
print("Mediana de ingresos:", datos["ingresos_mensuales"].median().round(0))

In [ ]:
# La moda aplica a variables categóricas
print("Tipo de crédito más frecuente:", datos["tipo_credito"].mode()[0])
print("Resultado más frecuente:      ", datos["resultado"].mode()[0])

**Pregunta para discutir:** la media de ingresos ¿es mayor o menor
que la mediana? ¿Qué nos dice eso sobre la distribución?

## 4.3 · Frecuencias y proporciones

In [ ]:
datos["tipo_credito"].value_counts()

In [ ]:
# En porcentaje
(datos["tipo_credito"].value_counts(normalize=True) * 100).round(1)

## 4.4 · Agrupar

`groupby` responde preguntas del tipo «¿cuánto o cuántos, por cada...?»
Es la operación más útil del análisis descriptivo.

In [ ]:
# ¿Cuántas solicitudes por ciudad?
datos.groupby("sucursal").size().sort_values(ascending=False)

In [ ]:
# ¿Cuál es el monto mediano por tipo de crédito?
datos.groupby("tipo_credito")["monto_solicitado"].median().sort_values(ascending=False).round(0)

In [ ]:
# Resumen completo por tipo
datos.groupby("tipo_credito")["monto_solicitado"].agg(
    ["count", "mean", "median", "min", "max"]
).round(0)

## 4.5 · Tablas cruzadas

`crosstab` cruza dos variables categóricas y cuenta las combinaciones.
Es el equivalente de una tabla dinámica en Excel.

In [ ]:
pd.crosstab(datos["tipo_credito"], datos["resultado"])

In [ ]:
# Como porcentaje de cada fila (cada tipo de crédito suma 100%)
(pd.crosstab(datos["tipo_credito"], datos["resultado"], normalize="index") * 100).round(1)

---
### Tarea 4.A · Para hacer en clase

**Pregunta 1:** Calcule la mediana del monto solicitado por sucursal.
¿Cuál ciudad pide los créditos más altos? ¿Tiene sentido?

In [ ]:
# Escriba su código aquí

**Pregunta 2:** Haga un `crosstab` entre `tipo_credito` y `resultado`.
¿Qué tipo tiene la mayor tasa de rechazo? ¿Y la menor?

In [ ]:
# Escriba su código aquí

**Pregunta 3:** ¿Hay diferencia en la tasa de aprobación entre quienes
tienen otros créditos (`SI`) y quienes no (`NO`)?

In [ ]:
# Escriba su código aquí

## 4.6 · Visualizaciones básicas

Un gráfico bien elegido comunica más rápido que una tabla.

In [ ]:
# Distribución de ingresos
plt.figure(figsize=(9, 3.5))
plt.hist(datos["ingresos_mensuales"], bins=40, edgecolor="white")
plt.xlabel("Ingresos mensuales (pesos)")
plt.ylabel("Número de solicitudes")
plt.title("Distribución de los ingresos declarados")
plt.tight_layout()
plt.show()

In [ ]:
# Solicitudes por tipo de crédito
conteo = datos["tipo_credito"].value_counts()
plt.figure(figsize=(8, 3.5))
plt.bar(conteo.index, conteo.values)
plt.ylabel("Número de solicitudes")
plt.title("Solicitudes por tipo de crédito")
plt.tight_layout()
plt.show()

In [ ]:
# Diagrama de caja del monto por tipo
plt.figure(figsize=(9, 3.5))
tipos_orden = ["EDUCACION", "LIBRE INVERSION", "VEHICULO", "VIVIENDA"]
datos_box = [datos.loc[datos["tipo_credito"] == t, "monto_solicitado"] for t in tipos_orden]
plt.boxplot(datos_box, tick_labels=tipos_orden)
plt.ylabel("Monto solicitado (pesos)")
plt.title("Distribución del monto por tipo de crédito")
plt.tight_layout()
plt.show()

## 4.7 · Ingeniería de características

Crear variables nuevas a partir de las existentes. Es lo que convierte
un dato registrado en un dato analizable.

### 4.7.1 · Relación monto / ingreso

Esta variable mide cuántas veces el monto solicitado supera el ingreso
mensual. Es un indicador básico de capacidad de pago.

In [ ]:
datos["ratio_monto_ingreso"] = (
    datos["monto_solicitado"] / datos["ingresos_mensuales"]
).round(1)

datos["ratio_monto_ingreso"].describe().round(1)

In [ ]:
# ¿Hay solicitudes con un ratio muy alto?
datos.loc[datos["ratio_monto_ingreso"] > 50,
          ["tipo_credito", "monto_solicitado", "ingresos_mensuales",
           "ratio_monto_ingreso", "resultado"]].head(10)

Los ratios más altos corresponden a créditos de vivienda. No son errores:
es la naturaleza del producto (plazos largos permiten montos altos).

### 4.7.2 · Rango de monto

In [ ]:
datos["rango_monto"] = pd.cut(
    datos["monto_solicitado"],
    bins=[0, 10_000_000, 30_000_000, 80_000_000, 300_000_000],
    labels=["Bajo", "Medio", "Alto", "Muy alto"]
)

datos["rango_monto"].value_counts()

### 4.7.3 · Antigüedad agrupada

In [ ]:
datos["grupo_antiguedad"] = pd.cut(
    datos["antiguedad_laboral"],
    bins=[-1, 1, 5, 15, 40],
    labels=["Nuevo (0-1)", "Junior (2-5)", "Senior (6-15)", "Veterano (16+)"]
)

datos["grupo_antiguedad"].value_counts()

### 4.7.4 · Extraer componentes de la fecha

In [ ]:
datos["anio"] = datos["fecha_solicitud"].dt.year
datos["mes"] = datos["fecha_solicitud"].dt.month

datos[["fecha_solicitud", "anio", "mes"]].head()

---
### Tarea 4.B · Para hacer en clase

**Pregunta 4:** Cree una variable que indique si el solicitante tiene
alta carga familiar (3 o más dependientes). Llámela `alta_carga`.
¿Los solicitantes con alta carga tienen una tasa de rechazo diferente?

In [ ]:
# Escriba su código aquí

**Pregunta 5:** Calcule el monto mediano por grupo de antigüedad.
¿Los más veteranos piden créditos más grandes?

In [ ]:
# Escriba su código aquí

**Pregunta 6:** ¿En qué mes del año se reciben más solicitudes?
Agrupe por `mes` y cuente. ¿Ve algún patrón estacional?

In [ ]:
# Escriba su código aquí

**Pregunta 7:** Proponga y construya una variable nueva que no esté
en esta lista. Explique en un comentario qué pregunta permite responder.

In [ ]:
# Escriba su código aquí

---
---
# Bloque 5 · Cierre

## 5.1 · Exportar el archivo final

In [ ]:
columnas_finales = [
    "id_solicitud", "fecha_solicitud", "anio", "mes",
    "sucursal", "tipo_credito", "monto_solicitado", "rango_monto",
    "plazo_meses", "ingresos_mensuales", "ratio_monto_ingreso",
    "antiguedad_laboral", "grupo_antiguedad",
    "nivel_educativo", "estado_civil",
    "numero_dependientes", "tiene_otros_creditos", "resultado"
]

final = datos[columnas_finales]
final.to_csv("solicitudes_credito_final.csv", index=False, encoding="utf-8")
final.shape

## 5.2 · Lo que aprendimos

Al terminar este cuaderno usted puede:

1. **Explorar** un archivo desconocido con `.shape`, `.info()`, `.describe()` y `.value_counts()`
2. **Diagnosticar** su calidad: completitud, consistencia, validez y unicidad
3. **Limpiar** en orden: texto → categorías → tipos → faltantes → duplicados
4. **Analizar** con agrupaciones, tablas cruzadas y visualizaciones básicas
5. **Enriquecer** con variables derivadas que habilitan nuevas preguntas
6. **Documentar** cada decisión para que otro pueda reproducir el trabajo

### Tarea final · Bitácora de limpieza

Documente en la celda siguiente el resumen completo de su trabajo:

- ¿Cuántos registros tenía el archivo original y cuántos quedaron?
- ¿Qué decisiones tomó ante los valores faltantes y por qué?
- ¿Qué variables nuevas creó y para qué sirven?
- ¿Qué haría distinto si este archivo alimentara un tablero en producción?

In [ ]:
# Escriba su bitácora aquí como comentarios

---
**Siguiente:** Módulo 2 — Introducción a Tableau.
El archivo `solicitudes_credito_final.csv` es el que conectarán en su
primer tablero.